In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '16'
# 模块导入
from src.train_utils import (
    set_seed
)
# 设置随机种子
set_seed(42)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")
import gc

使用设备: cuda


In [ ]:
## 加载模型配置

from pathlib import Path
print("加载配置...")
import yaml
model_config_path = '/root/lio/modelresearch/Configs/model_config.yaml'
train_config_path = '/root/lio/modelresearch/Configs/train_config.yaml'

with open(model_config_path, 'r', encoding='utf-8') as f:
    model_config = yaml.safe_load(f)
with open(train_config_path, 'r', encoding='utf-8') as f:
    train_config = yaml.safe_load(f)

model_version = model_config.get('model_version', 'multi_modal_model_1')
description = model_config.get('description')
print(f"model_version: {model_version}")
print(f"description: {description}")

## 保存配置
config_save_path = f"/root/lio/modelresearch/checkpoints/{model_version}/config"
config_save_path = Path(config_save_path)
# 确保保存目录存在
config_save_path.mkdir(exist_ok=True,parents=True)
## 保存model的config
model_config_save_path = f"{config_save_path}/model_config.yaml"
train_config_save_path = f"{config_save_path}/train_config.yaml"

with open(model_config_save_path, 'w') as f:
    yaml.dump(model_config, f, indent=4, sort_keys=False, allow_unicode=True)
with open(train_config_save_path, 'w') as f:
    yaml.dump(train_config, f, indent=4, sort_keys=False, allow_unicode=True)

加载配置...
model_version: TCN_multi_modal_model_1
description: 1.lobencode编码,TCN 2.lob数据是2*20,使用revin 4.感受野128


In [ ]:
# from Data_Pipeline.transform.lob_data_trans import transform_lob_data
# import polars as pl

# ## 划分数据集
# lob_dir = '/root/autodl-tmp/train_data/lob_data_100ms.parquet'
# # trade_dir = '/root/autodl-tmp/train_data/trade_data_100ms.parquet'
# labels_dir = '/root/autodl-tmp/train_data/labels_100ms.parquet'

# lob_data = pl.read_parquet(lob_dir)
# # trade_data = pl.read_parquet(trade_dir)
# labels = pl.read_parquet(labels_dir)

# split_date = '2025-11-30'
# ## 把split_date 转换为utc ms时间戳,utc的0时区
# from datetime import datetime, timezone
# # 直接构造 UTC 0 点时间戳（毫秒）
# dt_utc = datetime.strptime(split_date, "%Y-%m-%d").replace(
#     tzinfo=timezone.utc, hour=0, minute=0, second=0, microsecond=0
# )
# split_ts_ms = int(dt_utc.timestamp() * 1000)  # 转毫秒
# # lob_data


# train_lob = lob_data.filter(pl.col('time_bucket') < split_ts_ms)
# # train_trade = trade_data.filter(pl.col('time_bucket') < split_ts_ms)
# train_labels = labels.filter(pl.col('time_bucket') < split_ts_ms)


# val_lob = lob_data.filter(pl.col('time_bucket') >= split_ts_ms)
# # val_trade = trade_data.filter(pl.col('time_bucket') >= split_ts_ms)
# val_labels = labels.filter(pl.col('time_bucket') >= split_ts_ms)


# del lob_data
# # del trade_data
# gc.collect()



# lob_encoder_name = model_config.get('lob_encoder', {}).get('encoder_name', 'lob_encoder')
# ## 进行特征工程
# train_lob_data = transform_lob_data(train_lob,lob_encoder_name)
# # train_trade_data = train_trade.drop(['time_bucket']).to_numpy()
# train_labels_data = train_labels.drop(['time_bucket']).to_numpy()

# val_lob_data = transform_lob_data(val_lob,lob_encoder_name)
# # val_trade_data = val_trade.drop(['time_bucket']).to_numpy()
# val_labels_data = val_labels.drop(['time_bucket']).to_numpy()

# del train_lob
# # del train_trade
# del train_labels
# del val_lob
# # del val_trade
# del val_labels
# gc.collect()

# # data_dict = {'lob': train_lob,'trade': train_trade}


### 创建数据集

In [ ]:
# import yaml
# all_config = '/root/lio/Trade_LOB_MultiModal/Configs/experiment_config.yaml'
# with open(all_config, 'r') as f:
#     all_config = yaml.safe_load(f)
# import torch
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
## 划分训练集和验证集
## 创建数据集

# train_dict = {'lob': train_lob_data}
# train_dataset = MultiModalDataset(
#     train_dict, train_labels_data, augment=True
# )
# val_dict = {'lob': val_lob_data}
# val_dataset = MultiModalDataset(
#     val_dict, val_labels_data, augment=False
# )
# # =============================================================================
# # DataLoader配置
# # =============================================================================

# batch_size = 128
# num_workers = 0
# prefetch_factor = None
# pin_memory = False
# drop_last = True
# device = "cuda"

# # 创建 DataLoader
# from torch.utils.data import DataLoader
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=batch_size,
#     shuffle=True,
#     num_workers=num_workers ,  # GPU 上数据时 worker=0
#     # num_workers=0,
#     # collate_fn=multimodal_collate_fn,
#     prefetch_factor=prefetch_factor,
#     pin_memory=pin_memory,
#     drop_last=drop_last
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=batch_size,
#     shuffle=False,
#     # num_workers=num_workers if device == 'cpu' else 0,
#     num_workers=num_workers,
#     # collate_fn=multimodal_collate_fn,
#     # pin_memory=pin_memory and device == 'cpu',
#     prefetch_factor=prefetch_factor,
#     pin_memory=pin_memory,
#     drop_last=False
# )
from src.data.datamodule.data_loader import ETHUSDTDataLoaders
data_loaders = ETHUSDTDataLoaders()
 # 创建 DataLoader
print("创建 DataLoader...")
print(f"训练集大小: {len(data_loaders.train)}")
print(f"验证集大小: {len(data_loaders.val)}")

创建 DataLoader...
训练集大小: 224580
验证集大小: 69001


In [9]:
# ## 打印train_loader 的第一个
# print(next(iter(train_loader))[0].keys())

In [ ]:
for inputs,labels in data_loaders.train:
    print(inputs['lob'].shape,labels.shape)
    break

torch.Size([128, 6000, 2, 20]) torch.Size([128, 5])


### 模型创建

##### 1.多模态模型

In [11]:
# # 创建模型
# print("创建模型...")

# from Model import MultiModalTransformer
# # 根据数据情况调整配置
# lob_config = model_config.get('lob_encoder', {})
# trade_config = model_config.get('trade_encoder') 
# fusion_config = model_config.get('fusion', {})
# transformer_config = model_config.get('transformer', {})
# output_config = model_config.get('output_head', {})

# model = MultiModalTransformer(
#     lob_config=lob_config,
#     trade_config=trade_config,
#     fusion_config=fusion_config,
#     transformer_config=transformer_config,
#     output_config=output_config,
#     use_revin=True
# )


In [ ]:
from src.models.TCN_lobencoder import LOB_TCN
model = LOB_TCN(model_config,num_classes = 5)
# from torchinfo import summary
# # summary(model,(1,2,100,20))
# summary(
#     model,
#     input_data={"lob": inputs['lob']},
# )
model = model.to(device)

In [ ]:

import torch
import torch.nn as nn
from torchinfo import summary
# 3. 构造输入张量（维度需匹配配置）
for inputs,labels in data_loaders.train:
    lob_input = inputs['lob']
    # trade_input = inputs['trade']
    break
# 4. 封装模型：将字典输入转为位置参数（适配torchinfo）
class WrappedMultiModalModel(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.original_model = original_model
    
    def forward(self, lob, trade=None):
        # 构造模型需要的字典输入
        if trade is not None:
            inputs = {"lob": lob}
        else:
            inputs = {"lob": lob}
        return self.original_model(inputs)

# inputs = {'lob': lob_input}
# with torch.no_grad():
#     model(inputs)  # 这一步后，self.fusion 不再是 None
wrapped_model = WrappedMultiModalModel(model)

# 5. 调用summary（核心：传入输入张量列表，顺序匹配封装模型的forward参数）
summary(
    wrapped_model,
    input_data=[lob_input],  # 先lob，后trade
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    depth=5,  # 显示模型深度（层数）
    device="cuda"  # 若用GPU，改为"cuda"（需确保张量在GPU上）
)


Layer (type:depth-idx)                                                 Input Shape          Output Shape         Param #              Trainable
WrappedMultiModalModel                                                 [128, 6000, 2, 20]   [128, 5]             --                   True
├─LOB_TCN: 1-1                                                         [128, 6000, 2, 20]   [128, 5]             --                   True
│    └─RevIN2d: 2-1                                                    [128, 2, 6000, 20]   [128, 2, 6000, 20]   4                    True
│    └─LOBEncoder: 2-2                                                 [128, 2, 6000, 20]   [128, 100, 32]       --                   True
│    │    └─Sequential: 3-1                                            [128, 2, 6000, 20]   [128, 16, 6000, 10]  --                   True
│    │    │    └─Conv2d: 4-1                                           [128, 2, 6000, 20]   [128, 16, 6000, 10]  64                   True
│    │    │    └─Chann

In [14]:

# # 3. 构造输入张量（维度需匹配配置）
# batch_size = 2
# time_steps = all_config.get('data', {}).get('history_T', 3000)  # LOB/Trade的时间步必须一致
# lob_dim = model_config.get('lob_encoder', {}).get('in_channels', 4)
# # trade_dim = model_config.get('trade_encoder', {}).get('in_features', 12)
# trade_dim = 12
# lob_input = torch.randn(batch_size, lob_dim, time_steps, 10,device=device)  # (B,C,T,L) = (2,4,10,20)
# trade_input = torch.randn(batch_size, trade_dim, time_steps,device=device)    # (B,F,T) = (2,26,10)

# 4. 封装模型：将字典输入转为位置参数（适配torchinfo）
class WrappedMultiModalModel(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.original_model = original_model
    
    def forward(self, lob):
        # 构造模型需要的字典输入
        inputs = {"lob": lob}
        return self.original_model(inputs)

# inputs = {'lob': lob_input}
# with torch.no_grad():
#     model(inputs)  # 这一步后，self.fusion 不再是 None
wrapped_model = WrappedMultiModalModel(model)
model_summary_str = str(summary(
    wrapped_model,
    input_data=[lob_input],
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    depth=4,
    device="cuda"
))
import matplotlib.pyplot as plt
# 2. 用 Matplotlib 绘制文本图
plt.figure(figsize=(20, 25))  # 根据模型长度调整
plt.text(0.01, 0.99, model_summary_str, fontsize=10, verticalalignment='top', family='monospace')
plt.axis('off')
plt.tight_layout()

# 3. 保存图片
model_summary_save_path = f"/root/lio/modelresearch/checkpoints/{model_version}"
model_summary_save_path = Path(model_summary_save_path)
plt.savefig(os.path.join(model_summary_save_path, f'{model_version}_model_summary.png'), 
            dpi=150, bbox_inches='tight')
plt.close()
# 5. 调用summary（核心：传入输入张量列表，顺序匹配封装模型的forward参数）,保存为图片
# summary_img = summary(
#     wrapped_model,
#     input_data=[lob_input, trade_input],  # 先lob，后trade
#     col_names=["input_size", "output_size", "num_params", "trainable"],
#     col_width=20,
#     depth=5,  # 显示模型深度（层数）
#     device="cuda"  # 若用GPU，改为"cuda"（需确保张量在GPU上）
# )
# summary_img.savefig(os.path.join(self.output_dir, f'{variant_name}_model_summary.png'))

In [15]:
# 打印模型信息
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型参数量: {total_params:,} (可训练: {trainable_params:,})")

模型参数量: 91,401 (可训练: 91,401)


### 训练

In [16]:
import os
import torch

# # ===================== 日志关闭核心代码 (所有PyTorch版本通用) =====================
# # 关闭 torch.compile 的 AUTOTUNE 满屏刷屏日志 (重中之重)
# os.environ['TORCHINDUCTOR_PRINT_CONFIG'] = '0'
# os.environ['TORCHINDUCTOR_VERBOSE'] = '0'
# os.environ['TORCHINDUCTOR_AUTOTUNE_LOG'] = '0'
# # 关闭PyTorch编译器的所有警告/错误日志输出
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# os.environ['TORCH_LOGS'] = '0'

#'default'  , 'max-autotune'  , 'reduce-overhead'
print("Compiling model...")
model = torch.compile(model, mode='reduce-overhead') 

Compiling model...


In [17]:
from Train.trainer import Trainer
from Train.train_utils import (
    setup_optimizer,
    setup_scheduler,
    setup_loss_functions,
    # set_seed
)


# 设置优化器和调度器
optimizer = setup_optimizer(model, train_config.get('optimizer', {}))
scheduler = setup_scheduler(optimizer, train_config.get('scheduler', {}))

# 设置损失函数
loss_fn = setup_loss_functions(train_config.get('loss', {}))


In [18]:
# 创建训练器
Trainer_config = train_config.get('training', {})
trainer = Trainer(
    # model=model,
    # train_loader=train_loader,
    # val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    config=Trainer_config,
    # device=device,
    variant_name=model_version, ## 用于保存模型
    seed=42
)

# 开始训练和验证
print("=" * 50)
history = trainer.fit(model=model,train_loader=train_loader,val_loader=val_loader)

## 保存为pickle
import pickle
history_save_path = f"/root/lio/Trade_LOB_MultiModal/checkpoints/{model_version}"
history_save_path = Path(history_save_path)
with open(os.path.join(history_save_path, f'{model_version}_history.pkl'), 'wb') as f:
    pickle.dump(history, f)

print('history保存成功')

print("=" * 50)
print("训练完成!")

# print(f"最佳验证 F1 (Up/Down): {max(history['val_f1_updown']):.4f}")

开始训练，共 10 个 epoch
设备: cuda
AMP: True
梯度累积步数: 1
--------------------------------------------------


Epoch 1/10 (233.9s)
  Train Loss: 0.0000


TypeError: 'float' object is not subscriptable